# Deep Learning - Notebook 23
## Advanced Deep Learning & Frontier Models

Welcome to Module 23.

In Notebooks 19 through 22, we mastered the core architectures of Deep Learning: Multi-Layer Perceptrons, Convolutional Neural Networks (CNNs), and Transformers for Natural Language Processing (NLP).

Now, we step into the frontier of Artificial Intelligence: **Advanced & Generative Deep Learning**.

In this notebook, we explore four cutting-edge paradigms that push AI beyond standard prediction:
1. **Autoencoders & Variational Autoencoders (VAEs):** Unsupervised bottleneck networks for data compression, latent space generation, and industrial anomaly detection.
2. **Generative Adversarial Networks (GANs):** Competitive two-network games (Generator vs. Discriminator) for synthetic image generation.
3. **Reinforcement Learning (RL):** AI agents that learn optimal decision-making strategies through trial-and-error reward feedback loops.
4. **Graph Neural Networks (GNNs):** Neural networks designed to operate directly on relational graphs (social networks, molecular chemistry, and transaction networks).

We explore these frontier concepts using an anchor scenario:
* **Generative Image Synthesis, Anomaly Detection & AI Gaming Agent:** Building an industrial anomaly detector for manufacturing defects, training a generative adversarial framework for synthetic data generation, building a Q-learning agent for autonomous navigation, and modeling molecular networks with graph embeddings.

For every topic, we answer three questions:
* What is this in plain English?
* Why are we learning this?
* How is this used in Artificial Intelligence?

---
## Section 1: Autoencoders & Variational Autoencoders (VAEs)
### Data Compression & Anomaly Detection

### 1. What is this in plain English?
* **Autoencoder:** A neural network designed to copy its input to its output through a tight bottleneck:
  $$\text{Input } X \longrightarrow \text{Encoder} \longrightarrow \text{Latent Space (Bottleneck } Z) \longrightarrow \text{Decoder} \longrightarrow \text{Reconstructed } \hat{X}$$
  * **Encoder:** Compresses high-dimensional data (e.g., a 1024-pixel image) down to a compact bottleneck vector $Z$ (e.g., 16 numbers).
  * **Decoder:** Reconstructs the original data from the bottleneck vector $Z$.
  * **The Anomaly Detection Magic:** Train an Autoencoder ONLY on normal data (e.g., normal factory machine sounds or normal financial transactions). When an anomalous/corrupted record enters, the Autoencoder fails to reconstruct it, producing a high **Reconstruction Loss** ($\|X - \hat{X}\|^2$) that flags the anomaly instantly!

* **Variational Autoencoder (VAE):** Instead of encoding inputs into fixed points in latent space, a VAE encodes inputs into **probability distributions** (Mean $\mu$ and Standard Deviation $\sigma$). This allows you to sample random points from the latent space and generate brand-new, realistic synthetic data!

### 2. How is this used in AI?
* Industrial fault detection, image denoising, dimensionality reduction, and molecular drug discovery.

In [1]:
# AUTOENCODER ANOMALY DETECTION IN NUMPY / SCIKIT-LEARN

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

# GENERATE SYNTHETIC INDUSTRIAL SENSOR DATASET
np.random.seed(42)
n_normal = 500
n_anomalies = 20

# Normal Machine Operation (Correlated Sensor Readings)
sensor_1 = np.random.normal(50, 5, n_normal)
sensor_2 = sensor_1 * 2.0 + np.random.normal(0, 2, n_normal)
sensor_3 = 100 - sensor_1 + np.random.normal(0, 2, n_normal)

normal_sensors = np.column_stack([sensor_1, sensor_2, sensor_3])

# Abnormal Machine Failures (Anomalies)
anomaly_1 = np.random.uniform(10, 90, n_anomalies)
anomaly_2 = np.random.uniform(10, 90, n_anomalies)
anomaly_3 = np.random.uniform(10, 90, n_anomalies)

anomalous_sensors = np.column_stack([anomaly_1, anomaly_2, anomaly_3])

# Combine dataset
all_sensors = np.vstack([normal_sensors, anomalous_sensors])
labels = np.array([0] * n_normal + [1] * n_anomalies) # 0 = Normal, 1 = Anomaly

scaler = StandardScaler()
all_sensors_scaled = scaler.fit_transform(all_sensors)

# TRAIN AUTOENCODER ONLY ON NORMAL DATA (First 400 normal samples)
X_train_normal = all_sensors_scaled[:400]

# Bottleneck Architecture: 3 inputs -> 1 latent bottleneck -> 3 outputs
autoencoder = MLPRegressor(
    hidden_layer_sizes=(1,), # 1-dimensional bottleneck!
    activation="relu",
    max_iter=300,
    random_state=42
)
autoencoder.fit(X_train_normal, X_train_normal)

# CALCULATE RECONSTRUCTION LOSS ON ALL DATA
reconstructions = autoencoder.predict(all_sensors_scaled)
reconstruction_loss = np.mean((all_sensors_scaled - reconstructions) ** 2, axis=1)

# EVALUATE ANOMALY DETECTION THRESHOLD
threshold = np.percentile(reconstruction_loss[:n_normal], 98) # 98th percentile of normal loss
predicted_anomalies = (reconstruction_loss > threshold).astype(int)

print("--- Autoencoder Anomaly Detection Results ---")
print(f"Reconstruction Loss Threshold: {threshold:.4f}")
print(f"Normal Samples Average Reconstruction Loss:    {np.mean(reconstruction_loss[:n_normal]):.4f}")
print(f"Anomalous Samples Average Reconstruction Loss: {np.mean(reconstruction_loss[n_normal:]):.4f}")
print(f"Anomalies Correctly Flagged: {np.sum(predicted_anomalies[n_normal:] == 1)} out of {n_anomalies}")

--- Autoencoder Anomaly Detection Results ---
Reconstruction Loss Threshold: 2.0502
Normal Samples Average Reconstruction Loss:    0.3794
Anomalous Samples Average Reconstruction Loss: 12.1736
Anomalies Correctly Flagged: 20 out of 20


C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


---
## Section 2: Generative Adversarial Networks (GANs)
### Generator vs. Discriminator Adversarial Battles

### 1. What is this in plain English?
Introduced by Ian Goodfellow in 2014, a **Generative Adversarial Network (GAN)** consists of two neural networks locked in a competitive zero-sum game:

1. **The Generator ($G$):** The Art Forger. It takes a vector of random noise ($z$) and tries to generate realistic synthetic images (e.g., fake medical scans or synthetic human faces).
2. **The Discriminator ($D$):** The Police Detective. It inspects images and tries to classify whether they are **Real** (from the training dataset) or **Fake** (created by the Generator).

**The Adversarial Training Loop:**
* $D$ works to maximize its ability to spot fakes.
* $G$ works to minimize $D$'s ability to spot fakes (fooling $D$).
* As both networks train simultaneously, $G$ becomes so skilled that its synthetic outputs become indistinguishable from real data!

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

### 2. How is this used in AI?
* Generating synthetic training data for rare medical diseases.
* Image-to-image translation, super-resolution enhancement, and deepfake detection research.

In [3]:
# GAN ARCHITECTURE DEMONSTRATION IN PYTORCH / TENSORFLOW

try:
    import torch
    import torch.nn as nn

    # 1. THE GENERATOR NETWORK (Noise -> Synthetic Data)
    class Generator(nn.Module):
        def __init__(self, noise_dim, data_dim):
            super(Generator, self).__init__()
            self.net = nn.Sequential(
                nn.Linear(noise_dim, 32),
                nn.ReLU(),
                nn.Linear(32, data_dim),
                nn.Tanh() # Scales synthetic outputs to [-1, 1]
            )
        def forward(self, z):
            return self.net(z)

    # 2. THE DISCRIMINATOR NETWORK (Data -> Real/Fake Probability)
    class Discriminator(nn.Module):
        def __init__(self, data_dim):
            super(Discriminator, self).__init__()
            self.net = nn.Sequential(
                nn.Linear(data_dim, 32),
                nn.LeakyReLU(0.2),
                nn.Linear(32, 1),
                nn.Sigmoid() # Output probability: 1 = Real, 0 = Fake
            )
        def forward(self, x):
            return self.net(x)

    # INITIALIZE GAN NETWORKS
    noise_dim = 10
    data_dim = 4 # e.g., 4 synthetic sensor features

    G = Generator(noise_dim, data_dim)
    D = Discriminator(data_dim)

    # SIMULATE FORWARD PASS
    random_noise = torch.randn(5, noise_dim) # Batch of 5 noise vectors
    synthetic_data = G(random_noise)
    discriminator_verdict = D(synthetic_data)

    print("--- GAN Architecture Forward Pass ---")
    print("Synthetic Data Generated Shape:       ", synthetic_data.shape)
    print("Discriminator Real/Fake Verdicts:\n", discriminator_verdict.detach().numpy().round(4))

except ImportError:
    print("Note: PyTorch not installed. GAN architecture logic demonstrated above.")

--- GAN Architecture Forward Pass ---
Synthetic Data Generated Shape:        torch.Size([5, 4])
Discriminator Real/Fake Verdicts:
 [[0.5017]
 [0.4988]
 [0.5158]
 [0.5055]
 [0.5164]]


---
## Section 3: Reinforcement Learning (RL)
### Q-Learning & Agent-Environment Reward Loops

### 1. What is this in plain English?
Unlike Supervised Learning (which needs labeled answers) or Unsupervised Learning (which finds patterns in static data), **Reinforcement Learning** learns by **trial and error**.

**The RL Core Components:**
* **Agent:** The AI decision-maker (e.g., an autonomous driving agent or chess bot).
* **Environment:** The world the agent lives in.
* **State ($s$):** The current situation or position of the agent.
* **Action ($a$):** Moves the agent can make.
* **Reward ($r$):** Feedback from the environment (+100 for reaching the goal, -50 for crashing).

**Q-Learning & The Bellman Equation:**
The agent learns a $Q$-table $Q(s, a)$ that estimates the total future cumulative reward for taking action $a$ in state $s$. It updates its knowledge using the **Bellman Equation**:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

* $\alpha$: Learning rate.
* $\gamma$: Discount factor (how much the agent values future rewards vs. immediate rewards).

### 2. How is this used in AI?
* Autonomous robotics, self-driving vehicle path planning, algorithmic trading, and game-playing AI (AlphaGo, OpenAI Five).

In [4]:
# Q-LEARNING ALGORITHM IN PURE NUMPY
# Grid World Environment: 4 States (0, 1, 2, 3), 2 Actions (0 = Left, 1 = Right)
# Goal: Reach State 3 from State 0 to receive +100 Reward!

np.random.seed(42)

num_states = 4
num_actions = 2 # Left, Right

# Initialize Q-Table with zeros
Q_table = np.zeros((num_states, num_actions))

# Q-Learning Hyperparameters
alpha = 0.1   # Learning rate
gamma = 0.9   # Discount factor
episodes = 200

# Training Loop
for episode in range(episodes):
    state = 0 # Start at state 0
    done = False
    
    while not done:
        # Epsilon-greedy action selection (explore vs exploit)
        if np.random.rand() < 0.2:
            action = np.random.choice(num_actions) # Explore
        else:
            action = np.argmax(Q_table[state])     # Exploit best known Q-value
            
        # Environment step logic
        if action == 1: # Move Right
            next_state = min(state + 1, num_states - 1)
        else:          # Move Left
            next_state = max(state - 1, 0)
            
        # Reward function
        if next_state == 3:
            reward = 100.0 # Reached goal!
            done = True
        else:
            reward = -1.0  # Time penalty for each step
            
        # Bellman Equation Update
        best_future_q = np.max(Q_table[next_state])
        Q_table[state, action] += alpha * (reward + gamma * best_future_q - Q_table[state, action])
        
        state = next_state

print("--- Trained Q-Table (Expected Future Rewards) ---")
print("States x Actions [Left, Right]:")
print(np.round(Q_table, 2))
print("Optimal Strategy from State 0: Move Right repeatedly toward State 3!")

--- Trained Q-Table (Expected Future Rewards) ---
States x Actions [Left, Right]:
[[ 65.22  79.1 ]
 [ 65.54  89.  ]
 [ 66.31 100.  ]
 [  0.     0.  ]]
Optimal Strategy from State 0: Move Right repeatedly toward State 3!


---
## Section 4: Graph Neural Networks (GNNs)
### Nodes, Edges, & Message Passing on Relational Graphs

### 1. What is this in plain English?
Standard neural networks process grids (images) or sequences (text). But much of the real world is structured as **Graphs**:
* **Social Networks:** Users (**Nodes**) connected by friendships (**Edges**).
* **Financial Networks:** Bank accounts (**Nodes**) connected by transaction transfers (**Edges**).
* **Molecular Chemistry:** Atoms (**Nodes**) connected by chemical bonds (**Edges**).

**Graph Neural Networks (GNNs)** process non-Euclidean graph structures through a process called **Message Passing**. Every node gathers feature vectors from its connected neighbors, aggregates them, and updates its own representation vector!

### 2. How is this used in AI?
* Fraud detection in financial transaction graphs, drug molecule property prediction, and social network recommendation systems.

In [5]:
# SIMULATING GRAPH MESSAGE PASSING IN NUMPY

# Adjacency Matrix (A) for 4 Accounts: 1 if connected by transaction, 0 otherwise
A = np.array([
    [1, 1, 0, 0], # Account 0 connected to 0, 1
    [1, 1, 1, 0], # Account 1 connected to 0, 1, 2
    [0, 1, 1, 1], # Account 2 connected to 1, 2, 3
    [0, 0, 1, 1]  # Account 3 connected to 2, 3
])

# Node Feature Matrix (X): 4 Accounts x 2 Features [Risk_Score, Transaction_Volume]
X = np.array([
    [0.1, 100.0], # Acc 0
    [0.2, 150.0], # Acc 1
    [0.9, 500.0], # Acc 2 (High Risk Fraudulent Account!)
    [0.8, 450.0]  # Acc 3
])

# GRAPH CONVOLUTION MESSAGE PASSING: H = A @ X @ W
# Step 1: Aggregate neighbor features (A @ X)
aggregated_neighbor_features = A @ X

print("--- Graph Message Passing Aggregation ---")
print("Original Feature Matrix X:\n", X)
print("\nAggregated Neighbor Features (A @ X):\n", aggregated_neighbor_features)
print("Notice how Account 1 gathered risk signals from its neighbor Account 2!")

--- Graph Message Passing Aggregation ---
Original Feature Matrix X:
 [[1.0e-01 1.0e+02]
 [2.0e-01 1.5e+02]
 [9.0e-01 5.0e+02]
 [8.0e-01 4.5e+02]]

Aggregated Neighbor Features (A @ X):
 [[3.0e-01 2.5e+02]
 [1.2e+00 7.5e+02]
 [1.9e+00 1.1e+03]
 [1.7e+00 9.5e+02]]
Notice how Account 1 gathered risk signals from its neighbor Account 2!


---
## Hands-on Workshop: Advanced & Frontier Models

Complete these four practical challenges to build your advanced deep learning capabilities.

### Challenge 1: Autoencoder Reconstruction Error (Junior Level)

**Scenario:** Compute reconstruction loss to detect an abnormal sensor reading.

**Task Requirements:**
1. Given original data `X_original = np.array([[1.0, 2.0, 3.0], [1.0, 2.0, 3.0]])`.
2. Given reconstructed output `X_reconstructed = np.array([[1.1, 1.9, 3.0], [5.0, 8.0, 9.0]])`.
3. Compute Mean Squared Error for each row individually (`np.mean((orig - recon)**2, axis=1)`).
4. Print reconstruction error for both rows and identify which row is an anomaly.

In [4]:
# --- CHALLENGE 1: YOUR WORKSPACE ---




### Challenge 2: Bellman Equation Q-Value Update (Mid Level)

**Scenario:** Compute a single Bellman Equation update step for a gaming AI.

**Task Requirements:**
1. Current Q-value $Q(s, a) = 15.0$.
2. Reward $r = 10.0$.
3. Max future Q-value $\max Q(s', a') = 25.0$.
4. Learning rate $\alpha = 0.1$, Discount factor $\gamma = 0.9$.
5. Formula: $Q_{\text{new}} = Q(s, a) + \alpha [r + \gamma \max Q(s', a') - Q(s, a)]$.
6. Calculate and print $Q_{\text{new}}$.

In [3]:
# --- CHALLENGE 2: YOUR WORKSPACE ---




### Challenge 3: PyTorch Autoencoder Construction (Senior Level)

**Scenario:** Build a bottleneck PyTorch Autoencoder module.

**Task Requirements:**
1. Create a class `SimpleAutoencoder(nn.Module)`.
2. Encoder: Linear(10 inputs $\to$ 2 bottleneck outputs).
3. Decoder: Linear(2 bottleneck outputs $\to$ 10 reconstructed outputs).
4. Instantiate module and pass dummy tensor `x = torch.randn(1, 10)`. Print output tensor shape.

In [2]:
# --- CHALLENGE 3: YOUR WORKSPACE ---




### Challenge 4: Graph Convolution Matrix Multiplication (Architect Level)

**Scenario:** Compute a Graph Convolution message passing step $H = A \cdot X$.

**Task Requirements:**
1. Given 3-node Adjacency Matrix `A = np.array([[1, 1, 0], [1, 1, 1], [0, 1, 1]])`.
2. Given Node Feature Matrix `X = np.array([[10], [20], [30]])`.
3. Compute aggregated node features using matrix multiplication `@`.
4. Print aggregated feature vector.

In [1]:
# --- CHALLENGE 4: YOUR WORKSPACE ---




# Curriculum Summary & Conclusion

Congratulations. You have completed Notebook 23 and the entire Deep Learning curriculum!

Let us review what you have mastered across this Deep Learning Block:
1. **Notebook 19 (Fundamentals & Frameworks):** Neural network architecture, perceptrons, activation functions, loss functions, backpropagation gradients, optimizers (Adam), regularization (Dropout, BatchNorm), and model pipelines in both **TensorFlow/Keras** and **PyTorch**.
2. **Notebook 20 (Computer Vision & CNNs):** Convolutions, max pooling, classic vision architectures (ResNet, VGG), transfer learning, object detection (YOLO, IoU), and image segmentation.
3. **Notebook 21 (NLP Foundations & Embeddings):** Text preprocessing, TF-IDF, topic modeling (LDA), dense word embeddings (Word2Vec, GloVe, FastText), NER entity extraction, and sentiment routing pipelines.
4. **Notebook 22 (Modern NLP & Transformers):** Self-Attention mechanisms ($QK^T V$), Encoder-Decoder families (BERT, GPT, T5), Hugging Face tokenizers/pipelines, and LLM fine-tuning architecture.
5. **Notebook 23 (Advanced & Frontier Models):** Autoencoders for anomaly detection, VAEs, Generative Adversarial Networks (GANs), Reinforcement Learning (Q-Learning Bellman updates), and Graph Neural Networks (GNNs).

You have completed a comprehensive journey from Python basics and advanced mathematics to state-of-the-art Artificial Intelligence engineering. You possess the theoretical understanding, practical skills, and architectural knowledge to build, train, deploy, and explain production-grade AI systems!